# Initialization

In [1]:
from functions.PPTX_CHARTER_KT_THEME import *
from functions.LIFTROI_DATA_QUALITY_FUNCTIONS import *

# Input

In [2]:
# IMPORTANTE: Actualizar con la disponibilidad de datos 
flag_media = True
flag_sales = True
flag_oth = True 

In [3]:
# Carga de datasets 
data = pd.read_excel("Input.xlsx", sheet_name="DATA") 
hierarchy = pd.read_excel("Input.xlsx", sheet_name='HIERARCHY')

# Anticheat: convertir zeros a NAs 
data = data.replace(0, np.nan)

# Eliminación de periodos y breaks vacios 
data = data.dropna(subset=['Breaks','Period'])

# Apilamiento del set de datos 
stacked_data = data.melt(id_vars=('Period', 'Breaks')).copy()

# Union con jerarquia 
stacked_data = pd.merge(
    left=stacked_data, 
    right=hierarchy, 
    left_on='variable', 
    right_on='Variable Name',
    how='left')

# Input Diagnostics

In [4]:
check_non_numeric_columns(data, ['Breaks','Period'])

Data OK


In [5]:
check_hierarchy_inconsistency(stacked_data,'L0')

Stacked data OK


# Processing

In [ ]:
# Generación de descriptivos para identificar principales problemas de datos 

summary_describe = data.drop('Period', axis=1).describe().T

summary_describe['IQR'] = summary_describe['75%'] - summary_describe['25%']
summary_describe['Integrity'] = summary_describe['count'] / summary_describe['count'].max() * 100
summary_describe['Normalized Mean-Median'] = (summary_describe['mean'] - summary_describe['50%']) / summary_describe['mean']
summary_describe['Normalized STD-IQR'] = summary_describe['std'] - summary_describe['IQR'] / summary_describe['mean']

sheet_describe = summary_describe.round(4)

# Correlaciones 
sheet_correlations = data.fillna(0).corr(numeric_only=True)

if flag_media:

    # Diagnostico de integridad
    sheet_media_integrity = media_integrity_checker(data, hierarchy).round(3)

    # Frame de costo por actividad
    sheets_media_data = per_media_df_generator(data, hierarchy)
    sheet_cost_per_act = cost_per_activity_creator(sheets_media_data)
    sheet_cost_per_act_idx = sheet_cost_per_act / sheet_cost_per_act.mean() * 100
    sheet_describe_cpa = sheet_cost_per_act.describe().T.round(4)

    # Creamos la tabla resumen de spend por media 
    sheet_topline_media_spend = topline_spend_generator(stacked_data)
    sheet_topline_media_activity_sum = topline_activity_generator_sum(stacked_data)
    sheet_topline_media_activity_avg = topline_activity_generator_avg(stacked_data)

if flag_sales:

    # Creamos la tabla de resumen de ventas en revenue
    sheet_topline_sales_rev = topline_sales_revenue_generator(stacked_data)

    # Creamos las tablas de resumen de ventas por cada metrica de volumen
    sheets_topline_sales_vol = topline_sales_volume_generator(stacked_data)

if flag_oth:
    
    # Creamos las tablas de resumen de los otros esfuerzos que se agregan por promedio (precio, distribucion, etc)
    sheets_topline_efforts_avg = topline_efforts_avg_generator(stacked_data)

# Generación del topline completo 
# Variables agregadas por suma
sheet_full_sum_topline = add_change_delta(
    get_crosstab(
    stacked_df=stacked_data,
    index_vars=['L0','L1','L2','L3','L4','Metric'],
    column_vars=['Breaks'],
    value_var='value',
    filter_mask=(stacked_data['Aggregation']=='Sum'),
    agg_function='sum',
    str_separator=':'
    )
)

# Variables agregadas por promedio 
sheet_full_avg_topline = add_change_delta(
    get_crosstab(
    stacked_df=stacked_data,
    index_vars=['L0','L1','L2','L3','L4','Metric'],
    column_vars=['Breaks'],
    value_var='value',
    filter_mask=(stacked_data['Aggregation']=='Average'),
    agg_function='mean',
    str_separator=':'
    )
)

# Unión del topline 
sheet_full_topline = pd.concat([sheet_full_sum_topline, sheet_full_avg_topline])
sheet_full_topline.index = pd.MultiIndex.from_tuples(sheet_full_topline.index.str.split(':').map(tuple))

# Output

In [ ]:
forbidden_sheet_name_chars = r'[/\\\*\:\?\[\]]'

sheet_index_list = []

# Exportar los analisis en un Excel 
with pd.ExcelWriter('Output Data Quality Report.xlsx') as writer:
    
    sheet_full_topline.to_excel(writer, sheet_name='Topline')

    sheet_describe.to_excel(writer, sheet_name='Describe')

    if flag_media:
        sheet_media_integrity.to_excel(writer, sheet_name='Media integrity')
        sheet_index_list.append('Media integrity')

    sheet_correlations.to_excel(writer, sheet_name='Correlations')

    if flag_media:
        sheet_cost_per_act.to_excel(writer, sheet_name='Cost per activity')
        sheet_cost_per_act_idx.to_excel(writer, sheet_name='Cost per activity IDX')
        sheet_describe_cpa.to_excel(writer, sheet_name='Descriptivos CPA')
        sheet_index_list.append('Cost per activity')
        sheet_index_list.append('Cost per activity IDX')
        sheet_index_list.append('Descriptivos CPA')

    if flag_sales:
        sheet_topline_sales_rev.to_excel(writer, sheet_name='TL Sales $')
        sheet_index_list.append('TL Sales $')

        # Hojas de ventas en volumen 
        for sheet in range(len(sheets_topline_sales_vol)):
            sheets_topline_sales_vol[sheet][1].to_excel(writer, sheet_name=re.sub(forbidden_sheet_name_chars,'',sheets_topline_sales_vol[sheet][0])[:31])
            sheet_index_list.append(re.sub(forbidden_sheet_name_chars,'',sheets_topline_sales_vol[sheet][0])[:31])

    if flag_media:
        sheet_topline_media_spend.to_excel(writer, sheet_name='TL Media Spend')
        sheet_topline_media_activity_sum.to_excel(writer, sheet_name='TL Media Activty Sum')
        sheet_topline_media_activity_avg.to_excel(writer, sheet_name='TL Media Activty Avg')
        sheet_index_list.append('TL Media Spend')
        sheet_index_list.append('TL Media Activty Sum')
        sheet_index_list.append('TL Media Activty Avg')

    if flag_oth:
        # Hojas de esfuerzos en promedio  
        for sheet in range(len(sheets_topline_efforts_avg)):
            sheets_topline_efforts_avg[sheet][1].to_excel(writer, sheet_name=re.sub(forbidden_sheet_name_chars,'',sheets_topline_efforts_avg[sheet][0])[:31])
            sheet_index_list.append(re.sub(forbidden_sheet_name_chars,'',sheets_topline_efforts_avg[sheet][0])[:31])
    
    if flag_media:
        # Hojas de data por medio 
        for sheet in range(len(sheets_media_data)):
            sheets_media_data[sheet][1].to_excel(writer, sheet_name=re.sub(forbidden_sheet_name_chars,'',sheets_media_data[sheet][0])[:31])
            sheet_index_list.append(re.sub(forbidden_sheet_name_chars,'',sheets_media_data[sheet][0])[:31])

    data.to_excel(writer, sheet_name='DATA')
    stacked_data.to_excel(writer, sheet_name='STACKED DATA')

    sheet_index = sheet_index_creator(sheet_index_list)

    sheet_index.to_excel(writer, sheet_name='INDEX', index=False)

In [12]:
stacked_data.to_pickle('Output_Stacked_Data.pkl')

if flag_media:
    sheet_cost_per_act.to_pickle('Output_cost_per_act.pkl')